# Examine Post-Wildfire Evolution per Landcover Class

## Import Libs

In [ ]:
import os
import geopandas as gpd
import rasterio
from rasterio import features
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.dates import DateFormatter
from datetime import datetime
from pathlib import Path

## Define Paths

In [ ]:
notebook_dir = Path.cwd()
base_path = notebook_dir.parent.parent / "Data"
images_dir = os.path.join(base_path, "8_Images_with_Indices")

In [ ]:
# Get Classified Map
rf_map_path = os.path.join(base_path, "10_Landcover_Classification", "RF", "v2", "Classified_Full_Map_RF.shp") 
print("Load RF Classification Map...")
gdf = gpd.read_file(rf_map_path)


In [ ]:
# Load ref raster
ref_path = os.path.join(images_dir, "250821_7_channel.tif")
with rasterio.open(ref_path) as src_ref:
    out_shape = src_ref.shape
    transform = src_ref.transform

print("Rasterizing Vector map...")
rf_map = features.rasterize(
    [(geom, value) for geom, value in zip(gdf.geometry, gdf['Predicted'])],
    out_shape=out_shape,
    transform=transform,
    fill=0,      
    dtype='int16' 
)

## Post-Fire Evolution

In [ ]:
# Define order
flights = [
    ("2025-08-21", "250821"),
    ("2025-09-01", "250901"),
    ("2025-10-26", "251026"),
    ("2026-02-04", "260204"),
    ("2026-02-22", "260222")
]

# Define relevant classes
classes = {
    10: "Vineyard (Vital)",
    15: "Vineyard (Burned)",
    "10+15": "Vineyard (ALL 10+15)", # All Vines
    20: "Olive Tree",
    30: "Bare Soil",
    40: "Burned Area"
}

color_palette = {
    "Vineyard (Vital)": "forestgreen",
    "Vineyard (Burned)": "darkred",
    "Vineyard (ALL 10+15)": "limegreen",
    "Olive Tree": "darkolivegreen",
    "Bare Soil": "sandybrown",
    "Burned Area": "black"
}

In [ ]:
results = []

# --- Data Extraction ---
for date_str, prefix in flights:
    date_obj = datetime.strptime(date_str, "%Y-%m-%d")
    
    img_path = os.path.join(images_dir, f"{prefix}_7_channel.tif")
    if not os.path.exists(img_path):
        print(f"⚠️ Warning: Image {img_path} not found...")
        continue
        
    print(f"⏳ Processing flight {date_str}...")
    
    with rasterio.open(img_path) as src_img:
        # Band 5=ExG, 6=VARI, 7=NGRDI 
        exg = src_img.read(5)
        vari = src_img.read(6)
        ngrdi = src_img.read(7)
        
    nodata_mask = (exg == -9999.0)
    
    for cls_id, cls_name in classes.items():
        if cls_id == "10+15":
            # Mask for combined class
            class_mask = (rf_map == 10) | (rf_map == 15)
        else:
            class_mask = (rf_map == cls_id)
            
        valid_pixels = class_mask & ~nodata_mask
        
        if np.any(valid_pixels):
            results.append({
                "Date": date_obj,
                "Class": cls_name,
                "ExG": np.nanmean(exg[valid_pixels]),
                "VARI": np.nanmean(vari[valid_pixels]),
                "NGRDI": np.nanmean(ngrdi[valid_pixels])
            })

#  Save as df
df = pd.DataFrame(results)
print("\n✅ Extraktion done!")

# --- Generate Plots  ---
print("Generating Plots...")

# Format df
df_melt = pd.melt(df, id_vars=['Date', 'Class'], value_vars=['ExG', 'VARI', 'NGRDI'], 
                  var_name='Index', value_name='Mean Value')

# 3 Subplots 
fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharex=True)
indices = ['ExG', 'VARI', 'NGRDI']

for i, index_name in enumerate(indices):
    ax = axes[i]
    data_subset = df_melt[df_melt['Index'] == index_name]
    
    sns.lineplot(
        data=data_subset, 
        x='Date', 
        y='Mean Value', 
        hue='Class', 
        palette=color_palette,
        marker='o',
        linewidth=2,
        markersize=8,
        ax=ax
    )
    
    ax.set_title(f"Temporal Evolution of {index_name}", fontsize=15, fontweight='bold')
    ax.set_ylabel("Mean Index Value (Scaled)", fontsize=12)
    ax.set_xlabel("Flight Date", fontsize=12)
    
    date_form = DateFormatter("%b %Y")
    ax.xaxis.set_major_formatter(date_form)
    ax.tick_params(axis='x', rotation=45)
    
    ax.grid(True, linestyle='--', alpha=0.6)
    
    # Legende nur im letzten Plot anzeigen, um Platz zu sparen
    if i == 2:
        ax.legend(title="Landcover Class", bbox_to_anchor=(1.05, 1), loc='upper left')
    else:
        ax.get_legend().remove()

plt.tight_layout()

#  Save
save_dir = os.path.join(notebook_dir.parent.parent, "docs", "images")
os.makedirs(save_dir, exist_ok=True)
plt.savefig(os.path.join(save_dir, "temporal_evolution_indices.png"), dpi=300, bbox_inches='tight')

plt.show()